# Module 2 — Batch Data Ingestion with Snowflake

Module 2 is where the Tasty Bytes case study comes to life. You'll stand up the whole `tasty_bytes` database — raw tables, harmonized views, analytics views — and load them from public S3 files with `COPY INTO`. You'll also register a Snowflake Marketplace share for Weather Source data and walk through two smaller ingestion patterns (one purely in SQL, one that hands off to the Snowflake CLI on your laptop).

## Learning objectives
- Understand Snowflake's batch ingestion model (**stages + file formats + `COPY INTO`**).
- Create the multi-schema `tasty_bytes` database that every subsequent module builds on.
- Register a Marketplace share so external data is queryable as if it were native.
- Recognize when to load via a **named external stage** vs a **CLI-uploaded named stage** (both are valid; the choice depends on where the source file lives).

## Prerequisites
- `ACCOUNTADMIN` or equivalent to create databases, warehouses, integrations, and stages.
- Snowflake CLI installed locally if you plan to run the optional CLI upload exercise. See [Snowflake CLI docs](https://docs.snowflake.com/en/developer-guide/snowflake-cli-v2/installation/installation).

## What you'll build
- Database `tasty_bytes` with 4 schemas (`raw_pos`, `raw_customer`, `harmonized`, `analytics`) and ~8 tables.
- Two views (`harmonized.orders_v`, `analytics.orders_v`) and a customer-loyalty metrics view.
- File format `csv_ff` and external S3 stage `s3load`.
- Registered Marketplace share `frostbyte_weathersource`.
- Sample menu table loaded via `COPY INTO` from S3.
- (Optional CLI exercise) `sample_orders_cli` loaded from a file `PUT`-ed to a Snowflake internal stage.

## 1. Set your session context

In [ ]:
USE ROLE accountadmin;

## 2. Create the database, schemas, and ingestion warehouse

Everything downstream lives inside `tasty_bytes`. Four schemas separate raw data by source system (POS, customer), a harmonized layer for joined/enriched data, and a curated analytics layer that end users query.

In [ ]:
-- create tasty_bytes database
CREATE OR REPLACE DATABASE tasty_bytes;

-- create raw_pos schema
CREATE OR REPLACE SCHEMA tasty_bytes.raw_pos;

-- create raw_customer schema
CREATE OR REPLACE SCHEMA tasty_bytes.raw_customer;

-- create harmonized schema
CREATE OR REPLACE SCHEMA tasty_bytes.harmonized;

-- create analytics schema
CREATE OR REPLACE SCHEMA tasty_bytes.analytics;

In [ ]:
-- create warehouse for ingestion (large so the COPY INTO calls finish quickly)
CREATE OR REPLACE WAREHOUSE demo_build_wh
   WAREHOUSE_SIZE = 'xlarge'
   WAREHOUSE_TYPE = 'standard'
   AUTO_SUSPEND = 60
   AUTO_RESUME = TRUE
   INITIALLY_SUSPENDED = TRUE;

## 3. File format and external stage

A **file format** describes how Snowflake should parse raw files (CSV, delimiter, headers, etc.). A **stage** is a named location — internal to Snowflake or in cloud object storage — that `COPY INTO` reads from. Here we point a stage at a public Tasty Bytes S3 bucket.

In [ ]:
CREATE OR REPLACE FILE FORMAT tasty_bytes.public.csv_ff
type = 'csv';

CREATE OR REPLACE STAGE tasty_bytes.public.s3load
url = 's3://sfquickstarts/tasty-bytes-builder-education/'
file_format = tasty_bytes.public.csv_ff;

## 4. Raw POS and customer tables

DDL for every raw table the pipeline needs. Column types match the CSVs on the S3 stage. Some columns are typed loosely (`VARCHAR` for what look like numeric or timestamp fields) because the source data is dirty and will be cleaned in the harmonized layer.

In [ ]:
-- country table build
CREATE OR REPLACE TABLE tasty_bytes.raw_pos.country
(
   country_id NUMBER(18,0),
   country VARCHAR(16777216),
   iso_currency VARCHAR(3),
   iso_country VARCHAR(2),
   city_id NUMBER(19,0),
   city VARCHAR(16777216),
   city_population VARCHAR(16777216)
);

In [ ]:
-- franchise table build
CREATE OR REPLACE TABLE tasty_bytes.raw_pos.franchise
(
   franchise_id NUMBER(38,0),
   first_name VARCHAR(16777216),
   last_name VARCHAR(16777216),
   city VARCHAR(16777216),
   country VARCHAR(16777216),
   e_mail VARCHAR(16777216),
   phone_number VARCHAR(16777216)
);

In [ ]:
-- location table build
CREATE OR REPLACE TABLE tasty_bytes.raw_pos.location
(
   location_id NUMBER(19,0),
   placekey VARCHAR(16777216),
   location VARCHAR(16777216),
   city VARCHAR(16777216),
   region VARCHAR(16777216),
   iso_country_code VARCHAR(16777216),
   country VARCHAR(16777216)
);

In [ ]:
-- menu table build
CREATE OR REPLACE TABLE tasty_bytes.raw_pos.menu
(
   menu_id NUMBER(19,0),
   menu_type_id NUMBER(38,0),
   menu_type VARCHAR(16777216),
   truck_brand_name VARCHAR(16777216),
   menu_item_id NUMBER(38,0),
   menu_item_name VARCHAR(16777216),
   item_category VARCHAR(16777216),
   item_subcategory VARCHAR(16777216),
   cost_of_goods_usd NUMBER(38,4),
   sale_price_usd NUMBER(38,4),
   menu_item_health_metrics_obj VARIANT
);

In [ ]:
-- truck table build
CREATE OR REPLACE TABLE tasty_bytes.raw_pos.truck
(
   truck_id NUMBER(38,0),
   menu_type_id NUMBER(38,0),
   primary_city VARCHAR(16777216),
   region VARCHAR(16777216),
   iso_region VARCHAR(16777216),
   country VARCHAR(16777216),
   iso_country_code VARCHAR(16777216),
   franchise_flag NUMBER(38,0),
   year NUMBER(38,0),
   make VARCHAR(16777216),
   model VARCHAR(16777216),
   ev_flag NUMBER(38,0),
   franchise_id NUMBER(38,0),
   truck_opening_date DATE
);

In [ ]:
-- order_header table build
CREATE OR REPLACE TABLE tasty_bytes.raw_pos.order_header
(
   order_id NUMBER(38,0),
   truck_id NUMBER(38,0),
   location_id FLOAT,
   customer_id NUMBER(38,0),
   discount_id VARCHAR(16777216),
   shift_id NUMBER(38,0),
   shift_start_time TIME(9),
   shift_end_time TIME(9),
   order_channel VARCHAR(16777216),
   order_ts TIMESTAMP_NTZ(9),
   served_ts VARCHAR(16777216),
   order_currency VARCHAR(3),
   order_amount NUMBER(38,4),
   order_tax_amount VARCHAR(16777216),
   order_discount_amount VARCHAR(16777216),
   order_total NUMBER(38,4)
);

In [ ]:
-- order_detail table build
CREATE OR REPLACE TABLE tasty_bytes.raw_pos.order_detail
(
   order_detail_id NUMBER(38,0),
   order_id NUMBER(38,0),
   menu_item_id NUMBER(38,0),
   discount_id VARCHAR(16777216),
   line_number NUMBER(38,0),
   quantity NUMBER(5,0),
   unit_price NUMBER(38,4),
   price NUMBER(38,4),
   order_item_discount_amount VARCHAR(16777216)
);

In [ ]:
-- customer loyalty table build
CREATE OR REPLACE TABLE tasty_bytes.raw_customer.customer_loyalty
(
   customer_id NUMBER(38,0),
   first_name VARCHAR(16777216),
   last_name VARCHAR(16777216),
   city VARCHAR(16777216),
   country VARCHAR(16777216),
   postal_code VARCHAR(16777216),
   preferred_language VARCHAR(16777216),
   gender VARCHAR(16777216),
   favourite_brand VARCHAR(16777216),
   marital_status VARCHAR(16777216),
   children_count VARCHAR(16777216),
   sign_up_date DATE,
   birthday_date DATE,
   e_mail VARCHAR(16777216),
   phone_number VARCHAR(16777216)
);

## 5. Harmonized views

The harmonized layer stitches together the raw operational tables into a single order-level view (`orders_v`) — the go-to shape for any analytics that needs order + truck + menu + customer context in one query.

In [ ]:
-- orders_v view
CREATE OR REPLACE VIEW tasty_bytes.harmonized.orders_v
   AS
SELECT
   oh.order_id,
   oh.truck_id,
   oh.order_ts,
   od.order_detail_id,
   od.line_number,
   m.truck_brand_name,
   m.menu_type,
   t.primary_city,
   t.region,
   t.country,
   t.franchise_flag,
   t.franchise_id,
   f.first_name AS franchisee_first_name,
   f.last_name AS franchisee_last_name,
   l.location_id,
   cl.customer_id,
   cl.first_name,
   cl.last_name,
   cl.e_mail,
   cl.phone_number,
   cl.children_count,
   cl.gender,
   cl.marital_status,
   od.menu_item_id,
   m.menu_item_name,
   od.quantity,
   od.unit_price,
   od.price,
   oh.order_amount,
   oh.order_tax_amount,
   oh.order_discount_amount,
   oh.order_total
FROM tasty_bytes.raw_pos.order_detail od
JOIN tasty_bytes.raw_pos.order_header oh
   ON od.order_id = oh.order_id
JOIN tasty_bytes.raw_pos.truck t
   ON oh.truck_id = t.truck_id
JOIN tasty_bytes.raw_pos.menu m
   ON od.menu_item_id = m.menu_item_id
JOIN tasty_bytes.raw_pos.franchise f
   ON t.franchise_id = f.franchise_id
JOIN tasty_bytes.raw_pos.location l
   ON oh.location_id = l.location_id
LEFT JOIN tasty_bytes.raw_customer.customer_loyalty cl
   ON oh.customer_id = cl.customer_id;

In [ ]:
-- loyalty_metrics_v view
CREATE OR REPLACE VIEW tasty_bytes.harmonized.customer_loyalty_metrics_v
   AS
SELECT
   cl.customer_id,
   cl.city,
   cl.country,
   cl.first_name,
   cl.last_name,
   cl.phone_number,
   cl.e_mail,
   SUM(oh.order_total) AS total_sales,
   ARRAY_AGG(DISTINCT oh.location_id) AS visited_location_ids_array
FROM tasty_bytes.raw_customer.customer_loyalty cl
JOIN tasty_bytes.raw_pos.order_header oh
ON cl.customer_id = oh.customer_id
GROUP BY cl.customer_id, cl.city, cl.country, cl.first_name,
cl.last_name, cl.phone_number, cl.e_mail;

## 6. Analytics views

The analytics layer adds a curated, business-friendly touch on top of harmonized — in this case adding a materialized `DATE` column for easier grouping and a friendly comment for BI-tool discovery.

In [ ]:
-- orders_v view
CREATE OR REPLACE VIEW tasty_bytes.analytics.orders_v
COMMENT = 'Tasty Bytes Order Detail View'
   AS
SELECT DATE(o.order_ts) AS date, * FROM tasty_bytes.harmonized.orders_v o;

In [ ]:
-- customer_loyalty_metrics_v view
CREATE OR REPLACE VIEW tasty_bytes.analytics.customer_loyalty_metrics_v
COMMENT = 'Tasty Bytes Customer Loyalty Member Metrics View'
   AS
SELECT * FROM tasty_bytes.harmonized.customer_loyalty_metrics_v;

## 7. Load the raw tables with `COPY INTO`

Now we actually pull the data. We temporarily switch to `demo_build_wh` (XLarge) because these loads move meaningful volume and we want them to finish quickly. Each `COPY INTO` reads a subfolder on the S3 stage and lands the rows in the matching raw table.

In [ ]:
USE WAREHOUSE demo_build_wh;

In [ ]:
-- country table load
COPY INTO tasty_bytes.raw_pos.country
FROM @tasty_bytes.public.s3load/raw_pos/country/;

In [ ]:
-- franchise table load
COPY INTO tasty_bytes.raw_pos.franchise
FROM @tasty_bytes.public.s3load/raw_pos/franchise/;

In [ ]:
-- location table load
COPY INTO tasty_bytes.raw_pos.location
FROM @tasty_bytes.public.s3load/raw_pos/location/;

In [ ]:
-- menu table load
COPY INTO tasty_bytes.raw_pos.menu
FROM @tasty_bytes.public.s3load/raw_pos/menu/;

In [ ]:
-- truck table load
COPY INTO tasty_bytes.raw_pos.truck
FROM @tasty_bytes.public.s3load/raw_pos/truck/;

In [ ]:
-- customer_loyalty table load
COPY INTO tasty_bytes.raw_customer.customer_loyalty
FROM @tasty_bytes.public.s3load/raw_customer/customer_loyalty/;

In [ ]:
-- order_header table load (subset)
COPY INTO tasty_bytes.raw_pos.order_header
FROM @tasty_bytes.public.s3load/raw_pos/subset_order_header/;

In [ ]:
-- order_detail table load (subset)
COPY INTO tasty_bytes.raw_pos.order_detail
FROM @tasty_bytes.public.s3load/raw_pos/subset_order_detail/;

Drop the ingestion warehouse now that the loads are done — every subsequent module uses `compute_wh` instead.

In [ ]:
DROP WAREHOUSE demo_build_wh;

Quick end-to-end sanity check: preview the analytics `orders_v` (joined view) and count rows in a few key raw tables.

In [ ]:
SELECT * FROM tasty_bytes.analytics.orders_v LIMIT 5;

In [ ]:
import streamlit as st
from snowflake.snowpark.context import get_active_session

session = get_active_session()

tables = ["raw_pos.country", "raw_pos.franchise", "raw_pos.location",
          "raw_pos.menu", "raw_pos.truck", "raw_pos.order_header",
          "raw_pos.order_detail", "raw_customer.customer_loyalty"]

rows = []
for t in tables:
    n = session.sql(f"SELECT COUNT(*) AS c FROM tasty_bytes.{t}").collect()[0]["C"]
    rows.append({"table": t, "row_count": n})

st.subheader("Raw table row counts")
st.dataframe(rows, use_container_width=True)

## 8. Register the Weather Source Marketplace share

Weather Source publishes historical weather data as a Snowflake Data Cloud listing. Once you accept the listing in Snowsight (Data Products → Marketplace → search "Weather Source LLC: frostbyte") you'll have a database named `frostbyte_weathersource` available to query.

The query below is a quick smoke test — average temperature and total precipitation for cities in France over the last 7 days.

In [ ]:
USE DATABASE frostbyte_weathersource;
USE SCHEMA onpoint_id;

-- Returns the average temperature and total precipitation for cities in France
SELECT
  city_name,
  country,
  AVG(avg_temperature_air_2m_f) AS avg_temperature,
  SUM(tot_precipitation_in) AS total_precipitation
FROM
  history_day
WHERE
  date_valid_std >= DATEADD (DAY, -7, CURRENT_DATE) 
  AND country = 'FR'
GROUP BY
  city_name,
  country
ORDER BY
  total_precipitation DESC;

> **If the share isn't installed yet:** the two cells above will fail with "Database 'FROSTBYTE_WEATHERSOURCE' does not exist." Open Snowsight → **Data Products** → **Marketplace**, search **"Weather Source LLC: frostbyte"**, click **Get**, and accept the terms. Then re-run the cells.

## 9. Second ingestion pattern: load a single file from an external S3 stage

The section above loaded from a partitioned S3 folder. Sometimes you just want one specific file. Below we set up a fresh scratch database, point a stage at the general Tasty Bytes S3 bucket, list the files in a subfolder, and `COPY INTO` a specific menu file.

In [ ]:
USE ROLE accountadmin;
USE WAREHOUSE compute_wh;

CREATE OR REPLACE DATABASE load_data;
USE DATABASE load_data;
USE SCHEMA public;

-- Define the target table
CREATE OR REPLACE TABLE load_data.public.sample_menu_copy_into
(
   menu_id NUMBER(19,0),
   menu_type_id NUMBER(38,0),
   menu_type VARCHAR(16777216),
   truck_brand_name VARCHAR(16777216),
   menu_item_id NUMBER(38,0),
   menu_item_name VARCHAR(16777216),
   item_category VARCHAR(16777216),
   item_subcategory VARCHAR(16777216),
   cost_of_goods_usd NUMBER(38,4),
   sale_price_usd NUMBER(38,4),
   menu_item_health_metrics_obj VARIANT
);

-- Create external stage pointing to an AWS S3 bucket
-- Specify the file format
CREATE OR REPLACE STAGE load_data.public.blob_stage
url = 's3://sfquickstarts/tastybytes/'
file_format = (type = csv);

In [ ]:
-- Query the stage to note the file we'll load
LIST @load_data.public.blob_stage/raw_pos/menu/;

In [ ]:
-- Copy the file into the target table
COPY INTO load_data.public.sample_menu_copy_into
FROM @load_data.public.blob_stage/raw_pos/menu/;

-- Sanity check: query the newly loaded data
SELECT * FROM load_data.public.sample_menu_copy_into LIMIT 10;

## 10. Third ingestion pattern: load from a local file via Snowflake CLI

The two loads above pulled from external S3. The third pattern loads a file that lives **on your laptop** — you'll `PUT` it to a Snowflake-internal named stage from the terminal, then `COPY INTO` a table from the notebook.

### ⚠️ CLI handoff — leave the notebook for one command

Open a terminal and run:

```bash
cd modern-data-engineering-snowflake/module-2

snow stage create snowflake_cli_stage
snow stage copy sample_orders.csv @snowflake_cli_stage
```

The first command creates a Snowflake-internal named stage. The second `PUT`s `sample_orders.csv` (checked into this repo) up to that stage. Return to the notebook and run the next cell.

In [ ]:
USE DATABASE load_data;
USE SCHEMA public;

CREATE OR REPLACE TABLE load_data.public.sample_orders_cli ( 
    ORDER_ID NUMBER(38, 0), 
    TRUCK_ID NUMBER(38, 0), 
    LOCATION_ID NUMBER(38, 0), 
    CUSTOMER_ID NUMBER(38, 0), 
    DISCOUNT_ID VARCHAR, 
    SHIFT_ID NUMBER(38, 0), 
    SHIFT_START_TIME TIME, 
    SHIFT_END_TIME TIME, 
    ORDER_CHANNEL VARCHAR, 
    ORDER_TS TIMESTAMP_NTZ, 
    SERVED_TS VARCHAR, 
    ORDER_CURRENCY VARCHAR, 
    ORDER_AMOUNT NUMBER(38, 4), 
    ORDER_TAX_AMOUNT VARCHAR, 
    ORDER_DISCOUNT_AMOUNT VARCHAR, 
    ORDER_TOTAL NUMBER(38, 4) 
); 

CREATE OR REPLACE FILE FORMAT load_data.public.file_format_cli
  TYPE=CSV
    SKIP_HEADER=1
    FIELD_DELIMITER=','
    TRIM_SPACE=TRUE
    FIELD_OPTIONALLY_ENCLOSED_BY='"'
    REPLACE_INVALID_CHARACTERS=TRUE
    DATE_FORMAT=AUTO
    TIME_FORMAT=AUTO
    TIMESTAMP_FORMAT=AUTO;

In [ ]:
COPY INTO load_data.public.sample_orders_cli
FROM (SELECT $1, $2, $3, $4, $5, $6, $7, $8, $9, $10, $11, $12, $13, $14, $15, $16
  FROM @snowflake_cli_stage)
FILES = ('sample_orders.csv') 
FILE_FORMAT = ( FORMAT_NAME='load_data.public.file_format_cli')
ON_ERROR=ABORT_STATEMENT;

-- Sanity check
SELECT * FROM load_data.public.sample_orders_cli LIMIT 10;

## Recap

You practiced three batch-ingestion patterns:

| Pattern | Source | Best for |
|---|---|---|
| Partitioned folder on external S3 stage | Public/customer S3 bucket | Bulk load of many files (`tasty_bytes` main load) |
| Single file on external S3 stage | Known-good file in S3 | Targeted loads |
| Internal Snowflake stage via `snow stage copy` | File on your laptop | Prototyping, small local data |

Plus you registered a Marketplace share so external data is queryable exactly like native tables.

**The `tasty_bytes` database you built in section 2–7 is the foundation for Modules 3, 4, and 5** — keep it around.